# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haritharamadass/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)



## 1. Build the feature vector

### Feature vector

I build the feature vector using only search-performance information available during March 2026. April 2026 data is kept separate and is used only to define the future outcome.

The four model features are:

- `impressions_31d` — total March search impressions
- `ctr_31d` — March click-through rate
- `avg_position_31d` — impression-weighted average search position
- `active_gsc_days` — number of March days with available GSC data

Client and content identifiers are kept only for grouping and identification. They are not predictive features.

In [2]:
# ML-05 — Section 1
# Build the same public-safe feature vector used in the final capstone

from google.colab import userdata
import duckdb
import pandas as pd

# Get Hugging Face token safely from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN was not found. Add your Hugging Face READ token "
        "to Colab Secrets first."
    )

# Connect to DuckDB
con = duckdb.connect()

# Configure Hugging Face access
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face access configured successfully.")

# March = information available before prediction
MARCH = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

# April = future outcome window
APRIL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

# Build one row per anonymized client-content item
feature_data = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_31d,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_31d,

        SUM(gsc_avg_position * gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_impressions > 0
              AND gsc_avg_position > 0
        )
        /
        NULLIF(
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
                  AND gsc_impressions > 0
                  AND gsc_avg_position > 0
            ),
            0
        ) AS avg_position_31d,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS active_gsc_days

    FROM {MARCH}

    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS april_impressions,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS april_gsc_days

    FROM {APRIL}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.impressions_31d,

    100.0 * m.clicks_31d
        / NULLIF(m.impressions_31d, 0) AS ctr_31d,

    m.avg_position_31d,
    m.active_gsc_days,

    CASE
        WHEN a.april_impressions < 0.80 * m.impressions_31d
        THEN 1
        ELSE 0
    END AS declined_next_month

FROM march m

INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id

WHERE
    m.impressions_31d >= 100
    AND m.avg_position_31d IS NOT NULL
    AND m.avg_position_31d > 0
    AND m.active_gsc_days > 0
    AND a.april_gsc_days > 0
    AND a.april_impressions IS NOT NULL
""").df()

# Final feature list used by the capstone
FEATURES = [
    "impressions_31d",
    "ctr_31d",
    "avg_position_31d",
    "active_gsc_days"
]

TARGET = "declined_next_month"

# Keep only complete feature rows
feature_data = feature_data.dropna(
    subset=FEATURES + [TARGET]
).reset_index(drop=True)

X = feature_data[FEATURES].copy()
y = feature_data[TARGET].copy()

print("\nFeature vector built successfully.")
print("---------------------------------------------")
print(f"Rows: {len(feature_data):,}")
print(f"Clients: {feature_data['client_hash_id'].nunique():,}")
print(f"Content items: {feature_data['content_hash_id'].nunique():,}")

print("\nFeatures:")
for feature in FEATURES:
    print("-", feature)

print("\nTarget:", TARGET)

print("\nFeature preview:")
display(X.head())

print("\nMissing values:")
display(X.isna().sum().to_frame("missing"))

Hugging Face access configured successfully.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Feature vector built successfully.
---------------------------------------------
Rows: 100,893
Clients: 43
Content items: 100,893

Features:
- impressions_31d
- ctr_31d
- avg_position_31d
- active_gsc_days

Target: declined_next_month

Feature preview:


,impressions_31d,ctr_31d,avg_position_31d,active_gsc_days
0,6523.0,0.107313,6.893301,31
1,453.0,0.000000,3.433962,31
2,5630.0,0.106572,6.535346,31
3,4944.0,0.262945,7.435680,31
4,429.0,0.233100,3.983213,31



Missing values:


,missing
impressions_31d,0
ctr_31d,0
avg_position_31d,0
active_gsc_days,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

### Feature notes

All four predictive features are numerical and are calculated only from March 2026 data, so they are available before the April outcome window.

- **`impressions_31d`** — total Google Search Console impressions during March. Rows with fewer than 100 impressions are excluded so the analysis focuses on content with enough search visibility.
- **`ctr_31d`** — March click-through rate calculated from clicks and impressions. It measures how often search impressions resulted in clicks.
- **`avg_position_31d`** — impression-weighted average Google Search position during March. Invalid or missing position values are excluded.
- **`active_gsc_days`** — number of March days with available Google Search Console observations.

There are no categorical predictive features in the final feature vector. After applying the eligibility filters, the four model features have no missing values.

All of these features exist before the prediction moment. April performance is not included in the feature vector and is reserved only for defining the future outcome.

In [3]:
# ML-05 — Section 2
# Document each feature: meaning, type, missing handling,
# and whether it exists before the prediction moment.

feature_notes = pd.DataFrame([
    {
        "feature": "impressions_31d",
        "meaning": "Total GSC search impressions during March 2026",
        "type": "numeric",
        "missing_handling": "Rows with missing values are excluded; minimum 100 impressions required",
        "available_before_prediction": "YES"
    },
    {
        "feature": "ctr_31d",
        "meaning": "March clicks divided by March impressions, expressed as a percentage",
        "type": "numeric",
        "missing_handling": "Calculated only where impressions are greater than zero",
        "available_before_prediction": "YES"
    },
    {
        "feature": "avg_position_31d",
        "meaning": "Impression-weighted average GSC search position during March",
        "type": "numeric",
        "missing_handling": "Missing or invalid positions are excluded",
        "available_before_prediction": "YES"
    },
    {
        "feature": "active_gsc_days",
        "meaning": "Number of March days with available GSC data",
        "type": "numeric",
        "missing_handling": "At least one active GSC day is required",
        "available_before_prediction": "YES"
    }
])

print("Feature notes")
print("---------------------------------------------")
display(feature_notes)

print("\nCategorical predictive features: None")

print("\nMissing values in final feature vector:")
display(X.isna().sum().to_frame("missing"))

assert X.isna().sum().sum() == 0
assert set(feature_notes["available_before_prediction"]) == {"YES"}

print("\nCheck passed:")
print("- All predictive features are numerical.")
print("- All predictive features are available before April 2026.")
print("- No missing values remain in the final feature vector.")

Feature notes
---------------------------------------------


,feature,meaning,type,missing_handling,available_before_prediction
0,impressions_31d,Total GSC search impressions during March 2026,numeric,Rows with missing values are excluded; minimum...,YES
1,ctr_31d,"March clicks divided by March impressions, exp...",numeric,Calculated only where impressions are greater ...,YES
2,avg_position_31d,Impression-weighted average GSC search positio...,numeric,Missing or invalid positions are excluded,YES
3,active_gsc_days,Number of March days with available GSC data,numeric,At least one active GSC day is required,YES



Categorical predictive features: None

Missing values in final feature vector:


,missing
impressions_31d,0
ctr_31d,0
avg_position_31d,0
active_gsc_days,0



Check passed:
- All predictive features are numerical.
- All predictive features are available before April 2026.
- No missing values remain in the final feature vector.


## 3. The leakage hunt

### Leakage hunt

The model must use only information available at the March 2026 decision point. Any April performance field, future-window value, label-derived field, or direct copy of the target would create data leakage.

To demonstrate this, I deliberately create a leaked feature using the target itself. A model using this field would appear to perform extremely well because the answer has already been included in the inputs.

After showing this problem, I remove the leaked field and keep only the four March features used in the honest capstone model.

In [4]:
# ML-05 — Section 3
# Deliberately create a leaked feature, show why it is invalid,
# then remove it and restore the honest feature vector.

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ---------------------------------------------------------
# 1. Create a deliberately leaked feature
# ---------------------------------------------------------

leak_test = feature_data.copy()

# This is intentionally wrong:
# it directly copies the future label into the feature set.
leak_test["future_outcome_leak"] = leak_test[TARGET]

LEAKED_FEATURES = FEATURES + ["future_outcome_leak"]

X_leaked = leak_test[LEAKED_FEATURES]
y_leaked = leak_test[TARGET]

# Use a quick simple split only to demonstrate the leakage effect
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaked,
    y_leaked,
    test_size=0.20,
    random_state=42,
    stratify=y_leaked
)

leaked_model = LogisticRegression(max_iter=1000)

leaked_model.fit(X_train_l, y_train_l)

leaked_pred = leaked_model.predict(X_test_l)

leaked_accuracy = accuracy_score(y_test_l, leaked_pred)

print("Deliberate leakage experiment")
print("---------------------------------------------")
print("Leaked feature added: future_outcome_leak")
print(f"Accuracy with leaked feature: {leaked_accuracy:.3f}")

# ---------------------------------------------------------
# 2. Explain why this is invalid
# ---------------------------------------------------------

print("\nWhy this is leakage:")
print(
    "- future_outcome_leak is derived directly from the April outcome."
)
print(
    "- It would not be available when making the March prediction."
)
print(
    "- High performance from this feature would therefore be misleading."
)

# ---------------------------------------------------------
# 3. Remove the leaked feature
# ---------------------------------------------------------

HONEST_FEATURES = [
    "impressions_31d",
    "ctr_31d",
    "avg_position_31d",
    "active_gsc_days"
]

X_honest = feature_data[HONEST_FEATURES].copy()

assert "future_outcome_leak" not in X_honest.columns
assert TARGET not in X_honest.columns

print("\nLeakage removed successfully.")

print("\nFinal honest feature vector:")
for feature in HONEST_FEATURES:
    print("-", feature)

print("\nLeakage checks passed:")
print("- No April outcome field is used as a feature.")
print("- The target is not included in the feature vector.")
print("- Only March 2026 information remains.")

Deliberate leakage experiment
---------------------------------------------
Leaked feature added: future_outcome_leak
Accuracy with leaked feature: 1.000

Why this is leakage:
- future_outcome_leak is derived directly from the April outcome.
- It would not be available when making the March prediction.
- High performance from this feature would therefore be misleading.

Leakage removed successfully.

Final honest feature vector:
- impressions_31d
- ctr_31d
- avg_position_31d
- active_gsc_days

Leakage checks passed:
- No April outcome field is used as a feature.
- The target is not included in the feature vector.
- Only March 2026 information remains.


## 4. What I excluded and why

### What I excluded and why

I deliberately excluded fields that would either create leakage, identify private client information, or add unreliable inputs.

- **`client_hash_id`** — kept only for grouped validation so the same client does not appear in both training and test groups. It is not used as a predictive feature.
- **`content_hash_id`** — kept only as an anonymized content identifier for the review queue. It is not a predictive feature.
- **April performance fields** — excluded because April is the future outcome window. Using April values as features would leak future information.
- **`declined_next_month`** — this is the target label, not a feature.
- **Client names, domains, URLs, and raw search queries** — excluded for privacy and public-safety reasons.
- **GA4-derived metrics** — excluded from the final model because their coverage was lower for the selected March analysis window.
- **Label-derived or future-derived flags** — excluded because they could reveal information unavailable at the March decision point.

The final model therefore uses only four public-safe March search-performance features.

In [5]:
# ML-05 — Section 4
# Record the fields deliberately excluded from the predictive feature vector.

excluded_fields = pd.DataFrame([
    {
        "field": "client_hash_id",
        "reason": "Used only for client-grouped validation; not a predictive feature."
    },
    {
        "field": "content_hash_id",
        "reason": "Used only as an anonymized identifier for the review queue."
    },
    {
        "field": "April performance fields",
        "reason": "Future-window information; using it as input would create data leakage."
    },
    {
        "field": "declined_next_month",
        "reason": "This is the target label and must never be included in the feature vector."
    },
    {
        "field": "client names / domains / URLs / raw queries",
        "reason": "Excluded for privacy and public-safety reasons."
    },
    {
        "field": "GA4-derived metrics",
        "reason": "Excluded because coverage was lower for the selected March analysis window."
    },
    {
        "field": "label-derived or future-derived flags",
        "reason": "They could contain information unavailable at the March decision point."
    }
])

print("Fields deliberately excluded")
print("---------------------------------------------")
display(excluded_fields)

print("\nFinal predictive features:")
for feature in HONEST_FEATURES:
    print("-", feature)

# Safety checks
assert TARGET not in HONEST_FEATURES
assert "client_hash_id" not in HONEST_FEATURES
assert "content_hash_id" not in HONEST_FEATURES
assert len(HONEST_FEATURES) == 4

print("\nFinal safety check passed:")
print("- Target is excluded from predictive features.")
print("- Client and content identifiers are excluded from predictive features.")
print("- Future-window information is excluded.")
print("- Final model uses only four March 2026 features.")

Fields deliberately excluded
---------------------------------------------


,field,reason
0,client_hash_id,Used only for client-grouped validation; not a...
1,content_hash_id,Used only as an anonymized identifier for the ...
2,April performance fields,Future-window information; using it as input w...
3,declined_next_month,This is the target label and must never be inc...
4,client names / domains / URLs / raw queries,Excluded for privacy and public-safety reasons.
5,GA4-derived metrics,Excluded because coverage was lower for the se...
6,label-derived or future-derived flags,They could contain information unavailable at ...



Final predictive features:
- impressions_31d
- ctr_31d
- avg_position_31d
- active_gsc_days

Final safety check passed:
- Target is excluded from predictive features.
- Client and content identifiers are excluded from predictive features.
- Future-window information is excluded.
- Final model uses only four March 2026 features.
